# Freeman two-component decomposition: Python–C parity on ALOS-1

Compare the ground and volume scattering powers from Python and C-PolSARpro on the San Francisco ALOS-1 scattering matrix. Small numerical differences are expected; relative errors of a few percent are acceptable. Inspect absolute errors for powers near zero and investigate larger discrepancies.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import shutil
import subprocess
from pathlib import Path

import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

from polsarpro.decompositions import freeman2
from polsarpro.dev.io import read_psp_bin
from polsarpro.dev.metrics import summarize_metrics, visualize_errors
from polsarpro.io import open_netcdf_beam

executable = Path("/home/c_psp/Soft/bin/data_process_sngl/freeman_2components_decomposition.exe")
input_file = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_slc.nc")
input_dir = Path("/data/psp/SAN_FRANCISCO_ALOS1")
output_dir = Path("/data/psp/res/freeman2_alos1_c")
output_dir.mkdir(parents=True, exist_ok=True)

boxcar_size = (7, 7)

## Load the common input

Python reads the NetCDF scattering matrix while C reads the equivalent S2 bands. Apply the same validity mask to both inputs.

In [ ]:
S = open_netcdf_beam(input_file)
nrows, ncols = S.sizes["y"], S.sizes["x"]
mask_file = input_dir / "mask_valid_pixels.bin"
valid = read_psp_bin(mask_file) == 1
valid_da = xr.DataArray(valid, dims=("y", "x"), coords=S.coords)
S = S.where(valid_da)
S

## Run C-PolSARpro

Run this cell manually to collect the C runtime. One OpenMP thread avoids the race in the C image-wide span minimum and maximum updates. Output reading is performed later, outside the execution timing.

In [ ]:
%%time
c_files = {
    "ground": output_dir / "Freeman2_Ground.bin",
    "volume": output_dir / "Freeman2_Vol.bin",
}
memory_error_file = output_dir / "memory_error.txt"
command = [
    str(executable),
    "-id", str(input_dir),
    "-od", str(output_dir),
    "-iodf", "S2",
    "-nwr", str(boxcar_size[0]),
    "-nwc", str(boxcar_size[1]),
    "-ofr", "0",
    "-ofc", "0",
    "-fnr", str(nrows),
    "-fnc", str(ncols),
    "-errf", str(memory_error_file),
    "-mask", str(mask_file),
]
subprocess.run(command, env={**os.environ, "OMP_NUM_THREADS": "1"})

## Run Python

Run this cell manually to collect the Python runtime, including the lazy computation.

In [ ]:
%%time
with ProgressBar():
    out_py = freeman2(S, boxcar_size=boxcar_size).compute()

## Numerical comparison

C writes zero at masked pixels; convert these pixels to NaNs before comparison. The shared metrics report absolute errors and normalized errors, defined as the absolute difference divided by the sum of the two absolute powers. Check finite-value agreement separately because numerical metrics omit NaNs.

In [ ]:
# C does not write an output configuration file.
shutil.copy(input_dir / "config.txt", output_dir / "config.txt")
out_c = {
    name: np.where(valid, read_psp_bin(path), np.nan)
    for name, path in c_files.items()
}
for name in out_py.data_vars:
    assert out_c[name].shape == out_py[name].shape

summarize_metrics(out_py, out_c, short_titles=False, verbose=False)

In [ ]:
{
    name: {
        "finite_python": int(np.isfinite(out_py[name].values).sum()),
        "finite_C": int(np.isfinite(out_c[name]).sum()),
        "finite_mask_disagreements": int(
            (np.isfinite(out_py[name].values) != np.isfinite(out_c[name])).sum()
        ),
    }
    for name in out_py.data_vars
}

## Inspect spatial error patterns

Pay attention to image boundaries and masked regions when interpreting differences from the shared boxcar averaging.

In [ ]:
visualize_errors(out_py=out_py, out_c=out_c, sub_az=16)